In [2]:
import sys
sys.path.append("..")
from tools import search, vector_search, fulltext_search
from dotenv import load_dotenv
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langchain_deepseek import ChatDeepSeek
from pathlib import Path
import json
from tqdm.auto import tqdm
import pandas as pd
from rag import RAG
from pydantic import BaseModel, Field
from typing import Literal
import os
load_dotenv()

True

In [3]:
df_ground_truth = pd.read_csv("ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [4]:

def generate_agent_answer(rag, rec):
    question = rec["question"]
    answer_agent = rag.rag(question)
    json_file = Path("samples") / rec["filename"]
    with json_file.open("r", encoding="utf-8") as f:
        data = json.load(f)
        content = data["content"]

    return {
        "question": rec["question"],
        "title": rec["title"],
        "answer_agent": answer_agent,
        "tool_calls": json.dumps(rag.tool_calls),
        "filename": rec["filename"],
        "content": content,
    }

## Generate answers for different model

In [6]:
model = "gpt-5.4-mini"
Rag = RAG(model=model, provider="openai")
results = []
for rec in tqdm(ground_truth):
    results.append(generate_agent_answer(Rag, rec))
df_answers = pd.DataFrame(results)
df_answers.to_csv(f"agent-answers-{model}.csv", index=False)

  0%|          | 0/55 [00:00<?, ?it/s]

In [7]:
model = "gpt-5-nano-2025-08-07"
Rag = RAG(model=model, provider="openai")
results = []
for rec in tqdm(ground_truth):
    results.append(generate_agent_answer(Rag, rec))
df_answers = pd.DataFrame(results)
df_answers.to_csv(f"agent-answers-{model}.csv", index=False)

  0%|          | 0/55 [00:00<?, ?it/s]

In [8]:
model = "deepseek-v4-flash"
Rag = RAG(model=model, provider="deepseek")
results = []
for rec in tqdm(ground_truth):
    results.append(generate_agent_answer(Rag, rec))
df_answers = pd.DataFrame(results)
df_answers.to_csv(f"agent-answers-{model}.csv", index=False)

  0%|          | 0/55 [00:00<?, ?it/s]

In [9]:
model = "deepseek-v4-pro"
Rag = RAG(model=model, provider="deepseek")
results = []
for rec in tqdm(ground_truth):
    results.append(generate_agent_answer(Rag, rec))
df_answers = pd.DataFrame(results)
df_answers.to_csv(f"agent-answers-{model}.csv", index=False)

  0%|          | 0/55 [00:00<?, ?it/s]

In [38]:
class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [39]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.

Return a JSON object with exactly these four fields:
- answer_reasoning: your written reasoning about the answer quality
- answer_score: "good" or "bad"
- trajectory_reasoning: your written reasoning about the trajectory quality
- trajectory_score: "good" or "bad"
""".strip()

agent_judge_prompt = """
Question:
{question}

Original content (ground truth):
{content}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()


In [45]:
def evaluate(question, content, answer_agent, tool_calls):
    prompt = agent_judge_prompt.format(
        question=question,
        content=content,
        answer_agent=answer_agent,
        tool_calls=tool_calls,
    )
    llm = ChatDeepSeek(model="deepseek-v4-pro")
    structured_llm = llm.with_structured_output(AgentEvaluation, include_raw=True, method="json_mode")
    response = structured_llm.invoke(
        [
            ("system", agent_judge_instructions),
            ("user", prompt),
        ]
    )
    answer = response["parsed"]

    return {
        "question": question,
        "answer_agent": answer_agent,
        "answer_score": answer.answer_score,
        "answer_reasoning": answer.answer_reasoning,
        "trajectory_score": answer.trajectory_score,
        "trajectory_reasoning": answer.trajectory_reasoning,
    }


## Use judge to evaluate the answers generated by different models

In [47]:
def judge_agent_answers(model):
    df_answers = pd.read_csv(f"agent-answers-{model}.csv")
    answers = df_answers.to_dict(orient="records")
    agent_judge_evals = []
    for rec in tqdm(answers):
        eval_result = evaluate(
            question=rec["question"],
            content=rec["content"],
            answer_agent=rec["answer_agent"],
            tool_calls=rec["tool_calls"]
        )
        agent_judge_evals.append(eval_result)

    df_eval = pd.DataFrame(agent_judge_evals)
    df_eval.to_csv(f"agent-evaluation-{model}.csv", index=False)

In [48]:
models = ["gpt-5.4-mini", "gpt-5-nano-2025-08-07", "deepseek-v4-flash", "deepseek-v4-pro"]
for model in models:
    judge_agent_answers(model)

  0%|          | 0/55 [00:00<?, ?it/s]

  0%|          | 0/55 [00:00<?, ?it/s]

  0%|          | 0/55 [00:00<?, ?it/s]

  0%|          | 0/55 [00:00<?, ?it/s]

In [3]:
df_eval = pd.DataFrame(agent_judge_evals)

NameError: name 'agent_judge_evals' is not defined

In [56]:
def judge_agent_scores(df_eval):
    good_count = (df_eval["answer_score"] == "good").sum()
    total_count = len(df_eval)
    good_tool_count = (df_eval["trajectory_score"] == "good").sum()
    return good_count/total_count, good_tool_count/total_count

In [57]:
models = ["gpt-5.4-mini", "gpt-5-nano-2025-08-07", "deepseek-v4-flash", "deepseek-v4-pro"]
model_scores = {}
for model in models:
    df = pd.read_csv(f"agent-evaluation-{model}.csv")
    answer_score, tool_score = judge_agent_scores(df)
    model_scores[model] = (round(answer_score, 2), round(tool_score, 2))

pd.DataFrame(
    {"model": model_scores.keys(), "answer_score": [score[0] for score in model_scores.values()], "tool_score": [score[1] for score in model_scores.values()]}
).sort_values("answer_score", ascending=False).reset_index(drop=True)


,model,answer_score,tool_score
0,deepseek-v4-flash,0.98,0.85
1,deepseek-v4-pro,0.95,0.89
2,gpt-5.4-mini,0.87,0.98
3,gpt-5-nano-2025-08-07,0.85,1.00


`deepseek-v4-flash` has the best answer score and the lowest tool score. `gpt-5-nano-2025-08-07` has the lowest answer score and the highest tool score. This suggests that `deepseek-v4-flash` is the best model for generating answers in this evaluation.

In [4]:
df_eval= pd.read_csv(f"agent-evaluation-deepseek-v4-flash.csv")
df_bad = df_eval[df_eval["answer_score"] == "bad"].head()
df_bad

,question,answer_agent,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning
6,2. Why does Byzantine failure handling need mo...,"Based on the papers in the database, here is t...",bad,The agent's final answer gives consensus bound...,good,The search queries are relevant and contain im...
